# ML Pipeline - End-to-End Fraud Detection

Bu notebook fraud detection sürecini **gerçek bir production-ready pipeline** haline getiriyor. Raw CSV'den tahmine kadar tüm adımlar tek bir akışta:

1. **Raw Veri Yükleme** - `data/` klasöründen train_transaction + train_identity
2. **DType Optimizasyonu** - 01_fixingdtypes'daki bellek optimizasyonu
3. **Feature Engineering** - 04_FE'deki UID, aggregation ve tutar analizi
4. **Encoding & Preprocessing** - Label encoding, frequency encoding
5. **Model Eğitimi** - 05_ModelOptimization'dan Optuna parametreleri
6. **SHAP Feature Selection** - 06_ModelEvaluation'dan threshold=66

**Neden önemli?** Notebook'ta ayrı ayrı çalışan adımlar production'da birbirine bağlı değilse felaket kapıda. Training'de uygulanan bir preprocessing adımını inference'da unutursan, model tamamen farklı veri görür.

In [ ]:
import pandas as pd
import numpy as np
import warnings
import os
import json
import joblib

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
from sklearn.model_selection import GroupKFold
from lightgbm import LGBMClassifier

warnings.filterwarnings('ignore')
np.random.seed(42)

# ============================================================
# PIPELINE CONFIGURATION
# 05_ModelOptimization'dan Optuna best params
# 06_ModelEvaluation'dan SHAP+Native rank threshold
# ============================================================

OPTUNA_PARAMS = {
    'n_estimators': 500,
    'max_depth': 8,
    'learning_rate': 0.05,
    'num_leaves': 64,
    'min_child_samples': 100,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1
}

# SHAP + Native Importance Rank Sum Threshold
# 06_ModelEvaluation'da belirlendi: threshold <= 66 olan feature'lar kalır
SHAP_RANK_THRESHOLD = 66

# Excluded features (rank_sum > 66)
EXCLUDED_FEATURES = ['addr2_freq', 'card3_freq']

print("Pipeline Configuration:")
print(f"  SHAP Rank Threshold: {SHAP_RANK_THRESHOLD}")
print(f"  Excluded Features: {EXCLUDED_FEATURES}")
print(f"  LightGBM Params: n_estimators={OPTUNA_PARAMS['n_estimators']}, max_depth={OPTUNA_PARAMS['max_depth']}")
print("\nSetup complete")

## 1. Pipeline Fonksiyonları

Her adım fonksiyonlaştırılmış. Production'da bu fonksiyonları import edip kullanabilirsin.

In [ ]:
# ============================================================
# STEP 1: DATA LOADING (Raw CSV)
# ============================================================

def load_raw_data(data_dir='../../data', load_test=False):
    """
    Raw CSV dosyalarını yükle ve birleştir.
    01_fixingdtypes.ipynb'deki yükleme mantığı.
    
    Args:
        data_dir: Veri klasörü
        load_test: Test verisi de yüklensin mi?
    
    Returns:
        df: Birleştirilmiş DataFrame (_source kolonu ile)
    """
    print("Loading raw data...")
    
    # Transaction + Identity merge
    train_tr = pd.read_csv(f'{data_dir}/train_transaction.csv')
    train_id = pd.read_csv(f'{data_dir}/train_identity.csv')
    train_df = pd.merge(train_tr, train_id, on='TransactionID', how='left')
    train_df['_source'] = 'train'
    
    if load_test:
        test_tr = pd.read_csv(f'{data_dir}/test_transaction.csv')
        test_id = pd.read_csv(f'{data_dir}/test_identity.csv')
        test_df = pd.merge(test_tr, test_id, on='TransactionID', how='left')
        test_df['_source'] = 'test'
        df = pd.concat([train_df, test_df], axis=0, ignore_index=True)
    else:
        df = train_df
    
    print(f"  Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")
    print(f"  Memory: {df.memory_usage(deep=True).sum()/1024**2:.0f} MB")
    
    return df


# ============================================================
# STEP 2: DTYPE OPTIMIZATION
# ============================================================

def optimize_dtypes(df):
    """
    Bellek optimizasyonu: Float64 → Int, Object → Category
    """
    print("Optimizing data types...")
    initial_memory = df.memory_usage(deep=True).sum() / 1024**2
    
    float_cols = [col for col in df.columns if df[col].dtype == 'float64']
    for col in float_cols:
        non_null = df[col].dropna()
        if len(non_null) == 0:
            continue
        values = non_null.values
        if np.all(values == np.floor(values)):
            col_min, col_max = non_null.min(), non_null.max()
            if col_min >= -127 and col_max <= 127:
                df[col] = pd.array(df[col], dtype='Int8')
            elif col_min >= -32000 and col_max <= 32000:
                df[col] = pd.array(df[col], dtype='Int16')
            else:
                df[col] = pd.array(df[col], dtype='Int32')
    
    object_cols = [col for col in df.columns if df[col].dtype == 'object']
    for col in object_cols:
        nunique = df[col].nunique()
        cardinality_ratio = nunique / len(df)
        if cardinality_ratio < 0.5:
            df[col] = df[col].astype('category')
    
    final_memory = df.memory_usage(deep=True).sum() / 1024**2
    print(f"  Memory: {initial_memory:.0f} MB → {final_memory:.0f} MB")
    
    return df


# ============================================================
# STEP 3: FEATURE ENGINEERING
# ============================================================

def create_features(df):
    """
    Feature engineering: UID, aggregations, amount analysis
    """
    print("Creating features...")
    
    # Temporal features
    df['week_num'] = (df['TransactionDT'] // 604800).astype('int16')
    df['month_num'] = (df['week_num'] // 4).astype('int16')
    df['day'] = (df['TransactionDT'] // 86400).astype('int16')
    
    # UID Formula
    df['D1_filled'] = df['D1'].fillna(0)
    df['anchor_day'] = (df['day'] - df['D1_filled']).astype('int16')
    df['uid_card1_addr1'] = (df['card1'].astype(str) + '_' + 
                             df['addr1'].astype(str) + '_' + 
                             df['anchor_day'].astype(str))
    df['user_anchor_D1'] = df['TransactionDT'] - (df['D1_filled'].astype('float64') * 86400)
    df = df.drop('D1_filled', axis=1)
    
    # UID Aggregations
    uid_agg = df.groupby('uid_card1_addr1').agg({'TransactionAmt': ['mean', 'std', 'max', 'sum']})
    uid_agg.columns = ['uid_avg_amt', 'uid_std_amt', 'uid_max_amt', 'uid_total_amt']
    uid_agg = uid_agg.reset_index()
    df = df.merge(uid_agg, on='uid_card1_addr1', how='left')
    
    # Card1 Aggregations
    card1_agg = df.groupby('card1').agg({
        'TransactionAmt': ['mean', 'max'],
        'addr1': 'nunique'
    }).reset_index()
    card1_agg.columns = ['card1', 'card1_amt_mean', 'card1_amt_max', 'n_addr']
    df = df.merge(card1_agg, on='card1', how='left')
    
    # Frequency Encodings
    df['card1_FE'] = df.groupby('card1')['card1'].transform('count')
    df['addr1_FE'] = df.groupby('addr1')['addr1'].transform('count')
    
    # Amount Features
    dollars = df['TransactionAmt'].astype(int)
    cents = ((df['TransactionAmt'] - dollars) * 100).round().astype('int8')
    is_round = (cents == 0) & ((dollars % 100 == 0) | (dollars % 50 == 0) | (dollars % 10 == 0))
    is_psych = (cents == 99) | (cents == 95)
    df['is_random_amount'] = (~is_round & ~is_psych).astype('int8')
    
    # Fill Missing
    agg_features = ['uid_avg_amt', 'uid_std_amt', 'uid_max_amt', 'uid_total_amt',
                    'card1_amt_mean', 'card1_amt_max', 'n_addr', 'card1_FE', 'addr1_FE']
    for col in agg_features:
        if col in df.columns:
            df[col] = df[col].fillna(0)
    
    # Cleanup
    temp_cols = ['week_num', 'month_num', 'day', 'anchor_day', 'uid_card1_addr1']
    df = df.drop(columns=[c for c in temp_cols if c in df.columns], errors='ignore')
    df = df.loc[:, ~df.columns.duplicated()]
    
    print(f"  Created features. Shape: {df.shape}")
    return df


# ============================================================
# STEP 4: FEATURE SELECTION & ENCODING
# ============================================================

def select_and_encode_features(df, exclude_prefixes=['V', 'id_', 'D', 'M', 'C'],
                                le_dict=None, freq_maps=None):
    """
    Feature selection ve encoding.
    """
    print("Selecting and encoding features...")
    df = df.loc[:, ~df.columns.duplicated()]
    
    all_cols = df.columns.tolist()
    other_cols = [col for col in all_cols 
                  if not any(col.startswith(prefix) for prefix in exclude_prefixes)
                  and col not in ['TransactionID', 'isFraud', '_source']]
    
    keep_cols = other_cols.copy()
    if 'isFraud' in df.columns:
        keep_cols.append('isFraud')
    if 'TransactionDT' in df.columns:
        keep_cols.append('TransactionDT')
    if '_source' in df.columns:
        keep_cols.append('_source')
    
    keep_cols = list(dict.fromkeys(keep_cols))
    df_model = df[keep_cols].copy()
    
    # Label Encoding
    feature_cols = [col for col in other_cols if col != 'TransactionDT']
    categorical_cols = [col for col in feature_cols if df_model[col].dtype in ['object', 'category']]
    
    if le_dict is None:
        le_dict = {}
        for col in categorical_cols:
            if df_model[col].nunique() <= 50:
                le = LabelEncoder()
                df_model[col] = df_model[col].astype(str).replace({'nan': 'MISSING'})
                df_model[col] = le.fit_transform(df_model[col])
                le_dict[col] = le
    
    # Frequency Encoding
    if freq_maps is None:
        freq_maps = {}
        for col in categorical_cols:
            if df_model[col].nunique() > 50:
                freq_map = df_model[col].value_counts().to_dict()
                df_model[f'{col}_freq'] = df_model[col].map(freq_map).fillna(0)
                df_model = df_model.drop(col, axis=1)
                freq_maps[col] = freq_map
    
    # Fill NA
    numeric_cols = df_model.select_dtypes(include=[np.number]).columns.tolist()
    for col in numeric_cols:
        if df_model[col].isna().any():
            df_model[col] = df_model[col].fillna(-999)
    
    print(f"  Final shape: {df_model.shape}")
    return df_model, le_dict, freq_maps


print("Pipeline functions defined")

## 2. Pipeline Çalıştırma

In [ ]:
# ============================================================
# FULL PIPELINE EXECUTION
# ============================================================

# Step 1: Load raw data
df = load_raw_data(data_dir='../../data', load_test=False)

# Step 2: Optimize dtypes
df = optimize_dtypes(df)

# Step 3: Create features
df = create_features(df)

# Step 4: Select and encode features
df_model, le_dict, freq_maps = select_and_encode_features(df)

print(f"\n{'='*60}")
print("PIPELINE SUMMARY")
print(f"{'='*60}")
print(f"Final shape: {df_model.shape}")
print(f"Features: {df_model.shape[1] - 3}")
print(f"Memory: {df_model.memory_usage(deep=True).sum()/1024**2:.1f} MB")

## 3. Time-Based Train/Val/Test Split

In [ ]:
# ============================================================
# TIME-BASED SPLIT (60/20/20)
# ============================================================

df_model = df_model.sort_values('TransactionDT').reset_index(drop=True)
transaction_dt = df_model['TransactionDT'].copy()

y = df_model['isFraud']
X = df_model.drop(['isFraud', 'TransactionDT', '_source'], axis=1, errors='ignore')

n = len(X)
train_end = int(n * 0.6)
val_end = int(n * 0.8)

X_train, X_val, X_test = X.iloc[:train_end], X.iloc[train_end:val_end], X.iloc[val_end:]
y_train, y_val, y_test = y.iloc[:train_end], y.iloc[train_end:val_end], y.iloc[val_end:]

month_train = (transaction_dt.iloc[:train_end] // (86400 * 30)).astype('int16').values
month_val = (transaction_dt.iloc[train_end:val_end] // (86400 * 30)).astype('int16').values

print("TIME-BASED SPLIT (60/20/20)")
print(f"\n{'Set':<6} {'Samples':>12} {'Frauds':>10} {'Rate':>8}")
print(f"{'Train':<6} {len(X_train):>12,} {y_train.sum():>10,} {y_train.mean()*100:>7.2f}%")
print(f"{'Val':<6} {len(X_val):>12,} {y_val.sum():>10,} {y_val.mean()*100:>7.2f}%")
print(f"{'Test':<6} {len(X_test):>12,} {y_test.sum():>10,} {y_test.mean()*100:>7.2f}%")
print(f"\nFeatures: {X_train.shape[1]}")

## 4. SHAP Feature Selection

In [ ]:
# ============================================================
# SHAP + NATIVE RANK SUM FEATURE SELECTION
# ============================================================

def apply_shap_feature_selection(X_train, X_val, X_test, excluded_features, threshold=66):
    print(f"SHAP Feature Selection (threshold={threshold})")
    
    original_n = X_train.shape[1]
    selected_features = [f for f in X_train.columns if f not in excluded_features]
    
    X_train_sel = X_train[selected_features]
    X_val_sel = X_val[selected_features]
    X_test_sel = X_test[selected_features]
    
    print(f"  Original: {original_n} features")
    print(f"  Selected: {len(selected_features)} features")
    print(f"  Excluded: {excluded_features}")
    
    return X_train_sel, X_val_sel, X_test_sel, selected_features


X_train_sel, X_val_sel, X_test_sel, selected_features = apply_shap_feature_selection(
    X_train, X_val, X_test, 
    excluded_features=EXCLUDED_FEATURES,
    threshold=SHAP_RANK_THRESHOLD
)

print(f"\nFinal feature count: {len(selected_features)}")

In [ ]:
# ============================================================
# MODEL TRAINING
# ============================================================

print("Training LightGBM...")

model = LGBMClassifier(
    **OPTUNA_PARAMS,
    class_weight='balanced',
    random_state=42,
    verbose=-1,
    n_jobs=-1
)

model.fit(X_train_sel, y_train)

y_train_proba = model.predict_proba(X_train_sel)[:, 1]
y_val_proba = model.predict_proba(X_val_sel)[:, 1]
y_val_pred = model.predict(X_val_sel)

train_auc = roc_auc_score(y_train, y_train_proba)
val_auc = roc_auc_score(y_val, y_val_proba)

print(f"\n{'='*60}")
print("MODEL PERFORMANCE")
print(f"{'='*60}")
print(f"Train AUC: {train_auc:.4f}")
print(f"Val AUC:   {val_auc:.4f}")
print(f"Gap:       {train_auc - val_auc:.4f}")
print(f"\nPrecision: {precision_score(y_val, y_val_pred):.4f}")
print(f"Recall:    {recall_score(y_val, y_val_pred):.4f}")
print(f"F1 Score:  {f1_score(y_val, y_val_pred):.4f}")

## 5. Cross-Validation

In [ ]:
# ============================================================
# TIME-BASED GROUPKFOLD CV
# ============================================================

X_full = pd.concat([X_train_sel, X_val_sel], axis=0).reset_index(drop=True)
y_full = pd.concat([y_train, y_val], axis=0).reset_index(drop=True)
month_full = np.concatenate([month_train, month_val])

cv = GroupKFold(n_splits=3)

print(f"CV Strategy: GroupKFold with {cv.n_splits} splits\n")

cv_scores = []
for i, (train_idx, val_idx) in enumerate(cv.split(X_full, y_full, groups=month_full)):
    X_tr, X_vl = X_full.iloc[train_idx], X_full.iloc[val_idx]
    y_tr, y_vl = y_full.iloc[train_idx], y_full.iloc[val_idx]
    
    temp_model = LGBMClassifier(**{**OPTUNA_PARAMS, 'class_weight': 'balanced', 'random_state': 42, 'verbose': -1})
    temp_model.fit(X_tr, y_tr)
    score = roc_auc_score(y_vl, temp_model.predict_proba(X_vl)[:, 1])
    cv_scores.append(score)
    print(f"Fold {i+1}: {score:.4f}")

cv_scores = np.array(cv_scores)
print(f"\nCV AUC: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

## 6. Pipeline Kaydetme

In [ ]:
# ============================================================
# SAVE PIPELINE
# ============================================================

pipeline_dir = '../../models/pipeline'
os.makedirs(pipeline_dir, exist_ok=True)

pipeline_bundle = {
    'model': model,
    'le_dict': le_dict,
    'freq_maps': freq_maps,
    'selected_features': selected_features,
    'excluded_features': EXCLUDED_FEATURES,
    'optuna_params': OPTUNA_PARAMS,
    'shap_threshold': SHAP_RANK_THRESHOLD
}

joblib.dump(pipeline_bundle, f'{pipeline_dir}/fraud_pipeline_bundle.pkl')
joblib.dump(model, f'{pipeline_dir}/lgb_pipeline.pkl')

metadata = {
    'pipeline_version': '2.0',
    'metrics': {
        'val_auc': float(val_auc),
        'cv_auc_mean': float(cv_scores.mean()),
        'cv_auc_std': float(cv_scores.std())
    },
    'feature_selection': {
        'method': 'SHAP + Native Rank Sum',
        'threshold': SHAP_RANK_THRESHOLD,
        'n_features': len(selected_features)
    }
}

with open(f'{pipeline_dir}/pipeline_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"Saved to: {pipeline_dir}/")
for f_name in os.listdir(pipeline_dir):
    print(f"  {f_name}")

## 7. Production Inference

In [ ]:
# ============================================================
# PRODUCTION INFERENCE
# ============================================================

def predict_fraud_simple(X_processed, pipeline_path='../../models/pipeline/lgb_pipeline.pkl', threshold=0.5):
    model = joblib.load(pipeline_path)
    proba = model.predict_proba(X_processed)[:, 1]
    pred = (proba >= threshold).astype(int)
    risk = pd.cut(proba, bins=[0, 0.1, 0.3, 0.5, 0.7, 1.0],
                  labels=['Very Low', 'Low', 'Medium', 'High', 'Very High'])
    return {'predictions': pred, 'probabilities': proba, 'risk_categories': risk}


print("Testing inference...")
results = predict_fraud_simple(X_val_sel.head(10))
print(pd.DataFrame({
    'Prob': results['probabilities'].round(3),
    'Pred': results['predictions'],
    'Risk': results['risk_categories'],
    'Actual': y_val.head(10).values
}).to_string(index=False))

## 8. Sonuç

### Pipeline Yapısı

```
Raw CSV → DType Optimization → Feature Engineering → Encoding → SHAP Selection → Model → Tahmin
```

### Fonksiyonlar

| Fonksiyon | Açıklama |
|-----------|----------|
| `load_raw_data()` | Raw CSV yükle |
| `optimize_dtypes()` | Bellek optimizasyonu |
| `create_features()` | UID, aggregation features |
| `select_and_encode_features()` | Encoding |
| `apply_shap_feature_selection()` | Feature eleme |
| `predict_fraud_simple()` | Tahmin |

### Metrikler

- **Val AUC**: ~0.88
- **CV AUC**: Time-based GroupKFold ile stabil
- **Interpretable**: V, D, C, M black-box feature'lar yok